# Predicting dispel4py workflow performance with WfCommons

**The problem.** Someone asks "how long would this dispel4py workflow take with 500 tasks?"
Running it at 500 to find out is expensive.

**The approach.** Run it *small* a few times, learn its shape, generate a synthetic
500-task instance, and simulate that instead.

This notebook walks through `wfcommons.wfstream`, which does the learning and the
generating. It uses a real agentic climate-sensor workflow traced under three
dispel4py configurations.

---
### What you need to run this

1. dispel4py monitoring traces (the `monitor_*` artifacts from the
   `timed_simple` / `timed_multi` mappings) — set `TRACES` below.
2. This branch of WfCommons installed.

dispel4py itself is *not* needed: wfstream reads traces, it does not run workflows.

## Setup

`TRACES` is the only path you must change. Everything else is written to a
temporary directory, except the cooked recipe — see the note in step 3.

In [1]:
import json, logging, pathlib, shutil, tempfile, warnings, collections

# >>> POINT THIS AT YOUR TRACES <<<
TRACES = pathlib.Path("/home/taina/dispel4py_agentic_ai_traces")

WORK = pathlib.Path(tempfile.mkdtemp(prefix="wfstream-demo-"))
WFFORMAT, BUILD, SYNTHETIC = WORK/"wfformat", WORK/"build", WORK/"synthetic"

# WfChef fits distributions per task type and is chatty about the ones that fail;
# harmless here.
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

print("traces:", TRACES)
print("work:  ", WORK)

traces: /home/taina/dispel4py_agentic_ai_traces
work:   /tmp/wfstream-demo-948irnzx


## Step 0 — the three runs

The same workflow, traced three times. The `simple` run gives one process per PE;
the `multi` runs give several. **Both kinds matter, for different reasons**, and
we'll see why in steps 3 and 4.

dispel4py never records which real files the workflow read and wrote — the input
path arrives as a root input and the output path is a constant in the workflow
script — so we name them here.

In [2]:
TRACE_DIRS = ["monitoring_simple", "monitoring_multi_16", "monitoring_multi_32"]

INPUT_FILES = {
    "monitoring_simple":   ["sensor_data_agentic.json"],
    "monitoring_multi_16": ["sensor_data_parallel_100.json"],
    "monitoring_multi_32": ["sensor_data_parallel_100.json"],
}
OUTPUT_FILES = {
    "monitoring_simple":   ["agentic_sensor_results.jsonl"],
    "monitoring_multi_16": ["agentic_parallel_results.jsonl"],
    "monitoring_multi_32": ["agentic_parallel_results.jsonl"],
}

for d in TRACE_DIRS:
    artifacts = sorted(p.name for p in (TRACES/d).glob("monitor_*") 
                       if not p.name.endswith("Zone.Identifier"))
    print(f"{d:22} {len(artifacts)} artifacts")

# the two that carry the graph
print()
for pattern in ("monitor_concrete_shape_run*.json", "monitor_instances_run*.csv"):
    print(" ", next((TRACES/"monitoring_simple").glob(pattern)).name)

monitoring_simple      22 artifacts
monitoring_multi_16    34 artifacts
monitoring_multi_32    70 artifacts

  monitor_concrete_shape_run20260916T183309032089Z.json
  monitor_instances_run20260916T183309032089Z.csv


## Step 1 — traces → WfFormat

One WfFormat **task per PE instance** (`pe_id@rank`), runtimes from the instances
CSV, edges from the concrete shape.

dispel4py is a streaming system: PEs exchange data in memory over named
connections, not through files. Each connection becomes a zero-byte WfFormat
"file" so the dependency survives the format. The real files bookend it.

The task count goes in the filename. That is not cosmetic — step 3 depends on it.

In [3]:
from wfcommons.wfstream import convert_traces

instances = convert_traces.convert_traces(
    [TRACES/d for d in TRACE_DIRS], WFFORMAT,
    input_files=INPUT_FILES, output_files=OUTPUT_FILES)

for path in instances:
    print(convert_traces.describe(path))

monitoring_simple-7.json: 7 tasks, 7 streams, files=['agentic_sensor_results.jsonl', 'sensor_data_agentic.json']
monitoring_multi_16-13.json: 13 tasks, 13 streams, files=['sensor_data_parallel_100.json', 'agentic_parallel_results.jsonl']
monitoring_multi_32-31.json: 31 tasks, 31 streams, files=['sensor_data_parallel_100.json', 'agentic_parallel_results.jsonl']


In [4]:
# one converted task, in full
spec = json.loads(instances[0].read_text())["workflow"]["specification"]
task = next(t for t in spec["tasks"] if t["id"].startswith("NormalizeDataPE"))
print(json.dumps(task, indent=2))

{
  "name": "NormalizeDataPE1_ID0000004",
  "id": "NormalizeDataPE1_ID0000004",
  "parents": [
    "read0_ID0000006"
  ],
  "children": [
    "DeterministicPrecheckPE2_ID0000002"
  ],
  "inputFiles": [
    "read0:0.output"
  ],
  "outputFiles": [
    "NormalizeDataPE1:1.output"
  ]
}


Note `"id": "NormalizeDataPE1_ID000000n"` — WfChef reads the task *type* off the
id prefix, and the input/output "files" are the in-memory streams
(`read0:0.output` is "whatever instance `read0@0` wrote to its `output` port").

## Step 2 — cook a recipe

WfChef finds **microstructures**: subgraphs that repeat as the workflow grows. It
finds them by *comparing instances of different sizes*.

Watch the output: the `simple` run contributes **0 microstructures**. A single
pipeline with one instance per PE has nothing that repeats. This is why one trace
is never enough.

In [5]:
from wfcommons.wfstream import build_recipe

cooked = build_recipe.cook(WFFORMAT, BUILD, name="climate")
build_recipe.summarize(cooked)

  monitoring_multi_16-13: 6 microstructure(s) ResultWriterPE6x2, ActionExecutorPE5x2, DecisionMergePE3x2, ParallelLLMSensorAgentPE4x2, DeterministicPrecheckPE2x2, NormalizeDataPE1x2
  monitoring_multi_32-31: 6 microstructure(s) ResultWriterPE6x5, ActionExecutorPE5x5, DecisionMergePE3x5, ParallelLLMSensorAgentPE4x5, DeterministicPrecheckPE2x5, NormalizeDataPE1x5
  monitoring_simple-7: 0 microstructure(s) 
  error table:
    ,monitoring_multi_16-13,monitoring_multi_32-31
    monitoring_simple-7,0.09428090415820635,0.11670929301304388
    monitoring_multi_16-13,0.0,0.028569970957032224
    monitoring_multi_32-31,,0.0


The error table is WfChef's estimate of how well a recipe grown from the row's
instance approximates the column's — lower is better.

**Installing** copies the recipe into the `wfcommons` package so it can be
imported. This writes inside your installed WfCommons; the last cell restores it.

In [6]:
build_recipe.install(cooked)
print("installed:", cooked.name)

installed: wfchef_recipe_climate


## Step 3 — generate a synthetic instance

Here is the one place we **override WfChef**.

WfChef scales a workflow by replicating microstructures, and when it has none it
copies the whole graph — reader included. For a streaming workflow that is wrong:
scaling a dispel4py workflow means giving PEs `numprocesses > 1`, not running the
pipeline twice.

So `grow_pipeline` starts from the **simple** run and replicates non-source PEs
round-robin. Sources are never replicated — dispel4py runs a source in exactly one
process.

The simple run is found, not configured: it is the base graph with one instance
per PE, the only shape where replicating a PE *adds* parallelism instead of
multiplying parallelism that is already there.

In [7]:
from wfcommons.wfstream import streaming_recipe

print("base graphs:", sorted(streaming_recipe.base_graphs("climate")))
print("simple run :", streaming_recipe.simple_base_graph("climate"))

base graphs: ['monitoring_multi_16-13', 'monitoring_multi_32-31', 'monitoring_simple-7']
simple run : monitoring_simple-7


In [8]:
from wfcommons.wfstream import generate_workflows

synthetic = generate_workflows.generate([250], SYNTHETIC, name="climate")[0]

  climate-250.json: 250 tasks, 293 streams, widths=[1, 41, 42, 41, 42, 42, 41]
    ActionExecutorPE5=42, DecisionMergePE3=42, DeterministicPrecheckPE2=42, LLMSensorAgentPE4=41, NormalizeDataPE1=41, ResultWriterPE6=41, read0=1


Every generated instance is checked before it is kept: the metrics block must
match the real topology, and the graph must have **exactly one reader in exactly
one connected component**. Anything else is not a realisable dispel4py workflow,
and the file is deleted rather than left for something to train on.

In [9]:
report = generate_workflows.check_instance(synthetic)
print("levels :", report["widths"])
print("readers:", report["readers"], " components:", report["components"])
print("errors :", report["errors"] or "none")

levels : [1, 41, 42, 41, 42, 42, 41]
readers: 1  components: 1
errors : none


## Does it scale the way dispel4py does?

Generate a range of sizes and look at the instance counts per PE. A correct
streaming workflow keeps **one** reader at every scale and grows the rest evenly —
that is what `-n` would give each PE on a real run.

In [10]:
rows = []
for size in (50, 100, 200, 400):
    path = generate_workflows.generate([size], SYNTHETIC, name="climate")[0]
    s = json.loads(path.read_text())["workflow"]["specification"]
    rows.append((size, collections.Counter(t["id"].rsplit("_", 1)[0] for t in s["tasks"])))

pes = sorted(rows[0][1])
print(f"{'size':>6} | " + " ".join(f"{p[:14]:>15}" for p in pes))
print("-" * (8 + 16*len(pes)))
for size, counts in rows:
    print(f"{size:>6} | " + " ".join(f"{counts[p]:>15}" for p in pes))

  climate-50.json: 50 tasks, 59 streams, widths=[1, 8, 8, 8, 8, 9, 8]
    ActionExecutorPE5=9, DecisionMergePE3=8, DeterministicPrecheckPE2=8, LLMSensorAgentPE4=8, NormalizeDataPE1=8, ResultWriterPE6=8, read0=1
  climate-100.json: 100 tasks, 118 streams, widths=[1, 16, 17, 16, 17, 17, 16]
    ActionExecutorPE5=17, DecisionMergePE3=17, DeterministicPrecheckPE2=17, LLMSensorAgentPE4=16, NormalizeDataPE1=16, ResultWriterPE6=16, read0=1
  climate-200.json: 200 tasks, 234 streams, widths=[1, 33, 33, 33, 33, 34, 33]
    ActionExecutorPE5=34, DecisionMergePE3=33, DeterministicPrecheckPE2=33, LLMSensorAgentPE4=33, NormalizeDataPE1=33, ResultWriterPE6=33, read0=1
  climate-400.json: 400 tasks, 468 streams, widths=[1, 66, 67, 66, 67, 67, 66]
    ActionExecutorPE5=67, DecisionMergePE3=67, DeterministicPrecheckPE2=67, LLMSensorAgentPE4=66, NormalizeDataPE1=66, ResultWriterPE6=66, read0=1
  size |  ActionExecutor  DecisionMergeP  DeterministicP  LLMSensorAgent  NormalizeDataP  ResultWriterPE       

`read0` stays at 1. Everything else grows in step. That is the whole point of the
override.

---
## The two-call API

Everything above is one call. A registry (Laminar's, in our case) decides which
case applies and hands us the trace directories.

### Case A — a workflow the registry has not seen

Convert → cook → install → generate at the size the user asked about → simulate.

In [11]:
from wfcommons.wfstream import on_new_workflow

result = on_new_workflow(
    [TRACES/d for d in TRACE_DIRS],
    num_tasks=500,
    name="climate",
    wfformat_dir=WORK/"A"/"wfformat",
    build_dir=WORK/"A"/"build",
    synthetic_dir=WORK/"A"/"synthetic",
    input_files=INPUT_FILES, output_files=OUTPUT_FILES,
)
print("synthetic :", result["synthetic"].name)
print("simulation:", result["simulation"], "  <- simulate.py is the seam to fill")

  climate-500.json: 500 tasks, 584 streams, widths=[1, 83, 83, 83, 83, 84, 83]
    ActionExecutorPE5=84, DecisionMergePE3=83, DeterministicPrecheckPE2=83, LLMSensorAgentPE4=83, NormalizeDataPE1=83, ResultWriterPE6=83, read0=1
synthetic : climate-500.json
simulation: None   <- simulate.py is the seam to fill


### Case B — known workflow, a new size was run for real

Convert and **store** the new run, re-cook, stop. Nothing is generated and nothing
is simulated: the point is that the recipe is better the next time someone asks.

Below, a workflow registered from two runs, then a third arriving later.

In [12]:
from wfcommons.wfstream import on_new_size_run

B = WORK/"B"
on_new_workflow([TRACES/d for d in TRACE_DIRS[:2]], num_tasks=40, name="climate",
                wfformat_dir=B/"wfformat", build_dir=B/"build",
                synthetic_dir=B/"synthetic",
                input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("corpus after registration:", sorted(p.name for p in (B/"wfformat").glob("*.json")))

r = on_new_size_run([TRACES/TRACE_DIRS[2]], name="climate",
                    wfformat_dir=B/"wfformat", build_dir=B/"build",
                    input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("added:", [p.name for p in r["added"]], "| recipe re-cooked:", r["recipe"] is not None)

r2 = on_new_size_run([TRACES/TRACE_DIRS[2]], name="climate",
                     wfformat_dir=B/"wfformat", build_dir=B/"build",
                     input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("same run again ->", r2["skipped"], "| recipe re-cooked:", r2["recipe"] is not None)

  climate-40.json: 40 tasks, 48 streams, widths=[1, 6, 7, 6, 7, 7, 6]
    ActionExecutorPE5=7, DecisionMergePE3=7, DeterministicPrecheckPE2=7, LLMSensorAgentPE4=6, NormalizeDataPE1=6, ResultWriterPE6=6, read0=1
corpus after registration: ['monitoring_multi_16-13.json', 'monitoring_simple-7.json']
added: ['monitoring_multi_32-31.json'] | recipe re-cooked: True
same run again -> ['monitoring_multi_32'] | recipe re-cooked: False


## What is not built yet

`simulate.py` is a named stub. It should take a generated instance and return a
predicted makespan — the actual answer to "how long at 500 tasks?".

The reverse converter (WfFormat → a runnable dispel4py script) also lives outside
the tree right now. It would let you *check* a prediction by running the synthetic
workflow for real.

## Module map

| Module | Role |
|---|---|
| `dispel_fwd_converter` | traces → WfFormat (the parser) |
| `convert_traces` | rebuild the corpus from a set of traces |
| `update_traces` | add new runs to an existing corpus |
| `build_recipe` | cook + install a WfChef recipe |
| `streaming_recipe` | the scaling rule: replicate PEs, not pipelines |
| `generate_workflows` | write synthetic instances, validate them |
| `pipeline` | `on_new_workflow` / `on_new_size_run` |
| `simulate` | not built |
| `config` | defaults for the climate workflow |

## Cleanup

Restores the recipe that `install()` overwrote, and removes the temp directory.

In [13]:
import subprocess
subprocess.run(["git", "checkout", "--", "wfcommons/wfchef/recipes/"],
               cwd=pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "examples" else ".")
shutil.rmtree(WORK, ignore_errors=True)
print("restored the installed recipe; removed", WORK)

restored the installed recipe; removed /tmp/wfstream-demo-948irnzx
